In [ ]:
from nanover.recording import NanoverRecordingReader
from tutorials.experiments.keyframes import extract_keyframes

IN_PATH = "knot-tying-checkpoints-2.nanover.zip"
# IN_PATH = "trivial-checkpoints.nanover.zip"

with NanoverRecordingReader.from_path(IN_PATH) as reader:
    KEYFRAMES = extract_keyframes(reader)

KEYFRAMES

In [ ]:
## Setup runner
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="checkpoint replay")
imd_runner.load(0)

In [ ]:
simulation.use_pbc_wrapping = False

In [ ]:
from nanover.websocket import NanoverImdClient

client = NanoverImdClient.from_runner(imd_runner)
with client.root_selection.modify() as selection:
    selection.renderer = "cartoon"

In [ ]:
def notify_all(message):
    for command in imd_runner.app_server.commands:
        if "/notify" in command:
            imd_runner.app_server.run_command(command, dict(message=message))

In [ ]:
from smearmd import SmearAgent

KEYFRAME_INDEX = 0
AGENT: SmearAgent | None = None


def smear_next():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = min(len(KEYFRAMES) - 1, KEYFRAME_INDEX + 1)

    notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    smear_match(KEYFRAMES[KEYFRAME_INDEX])


def smear_back():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = max(0, KEYFRAME_INDEX - 1)

    notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    smear_match(KEYFRAMES[KEYFRAME_INDEX])


def smear_match(keyframe):
    global AGENT

    if AGENT is not None:
        AGENT.stop()

    AGENT = SmearAgent.from_runner(imd_runner)
    AGENT.speed = 5
    AGENT.keyframe = keyframe
    AGENT.start()


def smear_cancel():
    global AGENT, KEYFRAME_INDEX
    AGENT.stop()
    AGENT = None
    KEYFRAME_INDEX = 0

    notify_all(f"RESETTING AGENT")


imd_runner.app_server.register_command("user/smear/next", smear_next)
imd_runner.app_server.register_command("user/smear/back", smear_back)
imd_runner.app_server.register_command("user/smear/cancel", smear_cancel)

In [ ]:
from ipywidgets import Output
output = Output()
output

In [ ]:
with output:
    print("test")

In [ ]:
from nanover.jupyter.nglclient import NGLClient

client = NGLClient.from_runner(imd_runner)
client.view

In [ ]:
from nanover.jupyter.controls import show_runner_controls

show_runner_controls(imd_runner)